# NB 4.4 &mdash; El model del 97% que no serveix de res

**MP 5134** &mdash; UT4

*Dades: AEMET, estació de l'aeroport de Palma (2015-2023).*

---
### Què farem avui

Al NB 4.3 vam veure que, quan una classe és rara, el percentatge d'encerts enganya. Avui
ho portarem a l'extrem: predir la **pluja forta**, que a Palma només cau uns deu dies l'any.
Trobarem un model que encerta el 97% dels dies i no serveix per a res, aprendrem a fer
que el model miri els dies rars i acabarem amb un debat: quan és pitjor una falsa alarma,
i quan un cas que se'ns escapa?

Aquest notebook és més curt: dues pauses amb codi i una tercera de debat.

Quan acabis hauries de saber respondre aquestes preguntes:

- Per què un model que encerta el 97% dels dies pot no servir de res?
- Quina mètrica s'ha de mirar quan una classe és molt rara, i com es justifica?
- Què fa `class_weight="balanced"`?
- Com es decideix si és pitjor un fals positiu o un fals negatiu?

## 1. La pluja forta

Són les mateixes dades del NB 4.3. Ara la resposta és `pluja_forta_dema`: 1 si l'endemà
cauen més de 10 mm. Amb tanta aigua en un sol dia, els torrents poden sortir de mare i els
carrers s'inunden, i per això és un avís que interessa a Protecció Civil i als ajuntaments.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, precision_score,
                             recall_score)
from sklearn.model_selection import train_test_split

# Temps diari a l'aeroport de Palma (AEMET), sense els anys 2024 i 2025
URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT04-Classificacio_logistica_knn/aemet/pluja_palma.csv"

MESURES = ["tmed", "tmin", "tmax", "prec", "velmedia", "racha", "sol", "presMax", "presMin"]

df = pd.read_csv(URL_DADES)
print(f"Dies amb pluja forta l'endemà: {df['pluja_forta_dema'].sum()} "
      f"de {len(df)} ({df['pluja_forta_dema'].mean():.1%})")

Només 94 dies de 3.277, un 2,9%. És una classe molt més rara que la pluja del NB 4.3.

## 2. Un 97% d'encerts

Seguim la recepta de sempre, amb les nou mesures.

In [ ]:
X = df[MESURES]
y = df["pluja_forta_dema"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

model = LogisticRegression(max_iter=5000).fit(X_train, y_train)
referencia = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

print(f"Encerts del model:     {model.score(X_test, y_test):.1%}")
print(f"Encerts de referència: {referencia.score(X_test, y_test):.1%}")

ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test, display_labels=["no", "pluja forta"], cmap="Blues"
)
plt.title("Regressió logística")
plt.show()

El model encerta un 97,1% dels dies, exactament igual que la referència. La matriu diu
per què: **no avisa mai**. Dels 24 dies de pluja forta de la prova, no n'ha detectat cap.
Si algú us ensenyés aquest model presumint del 97%, us estaria venent un model que es
limita a dir sempre «no».

El model no és que no hagi après res: la probabilitat més alta que dona a un dia de la
prova és d'un 34%. El problema és que, amb tan pocs dies de pluja forta, cap no arriba al
llindar del 50%.

---
### Ara tu · Exercici 1: desmuntar el 97%

*Uns 30 minuts, en grup.*

**a)** Calculeu la precisió, el recall i la specificity del model i de la referència, com
al NB 4.3. Quan un model no avisa mai, la precisió no es pot calcular i scikit-learn dona
un avís: afegiu `zero_division=0` a `precision_score`. Quines mètriques delaten que el model
no serveix? Quina no ho delata?

**b)** Dibuixeu la corba ROC del model i de la referència al mateix gràfic. Quina AUC té el
model? Ha après alguna cosa, tot i no avisar mai?

**c)** La directora d'una empresa us diu: «Un 97% d'encerts! Ja el podem posar en marxa».
Escriviu-li una resposta de quatre o cinc línies, sense paraules tècniques, explicant per
què no.

**Compareu amb un altre grup.** Llegiu-vos les respostes. Quina convenceria més la directora?

In [ ]:
# Escriu aquí el teu codi

## 3. Fer que el model miri els dies rars

Hi ha dues maneres de fer que el model avisi més. La primera ja la coneixeu: **baixar el
llindar**, com al NB 4.3. La segona és dir-li, mentre entrena, que els dies rars compten
més. Ho fa el paràmetre `class_weight="balanced"`: com que hi ha unes 34 vegades més dies
sense pluja forta que amb, cada dia de pluja forta pesa com 34 dels altres.

In [ ]:
model_equilibrat = LogisticRegression(max_iter=5000, class_weight="balanced")
model_equilibrat.fit(X_train, y_train)

prediccio = model_equilibrat.predict(X_test)
print(f"Encerts:  {model_equilibrat.score(X_test, y_test):.1%}")
print(f"Recall:   {recall_score(y_test, prediccio):.1%}")
print(f"Precisió: {precision_score(y_test, prediccio):.1%}")

ConfusionMatrixDisplay.from_estimator(
    model_equilibrat, X_test, y_test, display_labels=["no", "pluja forta"], cmap="Blues"
)
plt.title("Regressió logística amb class_weight='balanced'")
plt.show()

Ara el model detecta 11 dels 24 dies de pluja forta, un recall del 45,8%. El preu és alt:
dona 216 avisos i només 11 encerten, una precisió del 5,1%. I els encerts baixen al 73,4%.
**El model és pitjor segons l'accuracy, però és l'únic dels dos que serveix per a alguna
cosa.**

Una precisió del 5% sembla molt poca, però avisant a l'atzar seria del 2,9%: el model hi
veu gairebé el doble. Hi ha problemes així de difícils, i amb les mesures d'un sol dia en un sol
lloc no podem fer gaire més. Explicar-ho amb sinceritat també és part de la feina.

---
### Ara tu · Exercici 2: un avís per a Protecció Civil

*Uns 40 minuts, en grup.*

Protecció Civil vol un avís de pluja forta per decidir si posa les brigades de guàrdia
l'endemà. Una guàrdia de més costa hores de feina; una pluja forta sense brigades pot
acabar amb carrers inundats i gent atrapada.

**a)** Amb `model_equilibrat`, ompliu la taula. Necessitareu `predict_proba`, com al NB 4.3.

| Llindar | Avisos | Recall | Precisió |
|---|---|---|---|
| 0,3 | | | |
| 0,5 | | | |
| 0,7 | | | |
| 0,8 | | | |

**b)** Trieu un llindar per a Protecció Civil. Quantes guàrdies de més hi hauria a l'any, més
o menys? La prova és d'uns 820 dies, poc més de dos anys.

**c)** A la prova només hi ha 24 dies de pluja forta: cada un que s'encerta o s'escapa mou el
recall més de 4 punts. Torneu a fer la partició amb `random_state=7` i `random_state=0` i
mireu si el vostre llindar continua donant un recall semblant.

**d)** Escriviu la **justificació de la mètrica** en un paràgraf: per què no podeu fer servir
el percentatge d'encerts, quina mètrica heu fet servir per triar el llindar i per què és la
que importa a Protecció Civil.

**Compareu amb un altre grup.** Heu triat la mateixa mètrica? I el mateix llindar?

In [ ]:
# Escriu aquí el teu codi

## 4. Quin error és pitjor?

Tots els classificadors de la unitat s'han equivocat de dues maneres: avisant quan no calia
(**fals positiu**) o callant quan calia avisar (**fals negatiu**). Quina de les dues és
pitjor no ho diu cap mètrica: depèn de les conseqüències. Per això, abans de triar la mètrica,
cal saber qui farà servir el model i què li passa quan s'equivoca.

---
### Ara tu · Exercici 3: debat

*Uns 40 minuts: 25 en grup i 15 de posada en comú amb tota la classe.*

Per a cada un d'aquests classificadors, ompliu la taula en grup:

| Classificador | Classe positiva | Què és un fals positiu? | Què és un fals negatiu? | Quin és pitjor? | Llindar alt o baix? |
|---|---|---|---|---|---|
| Prova de cribratge d'una malaltia greu | | | | | |
| Detector de pagaments fraudulents d'un banc | | | | | |
| Filtre de correu brossa | | | | | |
| Un d'inventat per vosaltres | | | | | |

Després, prepareu-vos per defensar a la posada en comú una de les respostes, la que més us
hagi fet discutir dins el grup.

Algunes preguntes per animar el debat:

- En el cribratge, què passa després d'un positiu? Canvia això quin error és pitjor?
- En el filtre de correu, què és més greu: trobar un correu brossa a la safata o no veure el
  correu que us convoca a una entrevista de feina?
- Hi ha cap cas en què els dos errors siguin igual de dolents?

## 5. El que ens emportem de la UT4

En aquesta unitat hem après a classificar amb dos models, la **regressió logística** i el
**k-NN**, sempre amb la mateixa recepta: preparar, partir amb `stratify`, entrenar, comparar
amb la referència i predir. I hem après a desconfiar del percentatge d'encerts: quan una
classe és rara, cal mirar la **matriu de confusió**, la **precisió** i el **recall**, triar
el **llindar** segons el client i comparar models amb la **corba ROC**.

Per sobre de tot, una idea: **la mètrica no la tria el model, la tria el problema**. Abans de
mesurar res, cal saber quin error surt més car.

A la UT5 farem servir la mateixa recepta i les mateixes mètriques amb els arbres de decisió.